# Orislop four-worker rolling dataset + Temporal MoE cache

Open this notebook in four Colab runtimes. Set `WORKER_ID` to 0, 1, 2, or 3 in each runtime. The workers process separate <=50 GiB batches and write verified frozen-expert caches to the same Google Drive. Only worker 0 should run the final merge/training cell after status is complete.

In [ ]:
WORKER_ID = 0  # change to 0, 1, 2, or 3 in each runtime
CREATE_PLAN = False  # true only once, after global-splits is on Drive
REPO_URL = 'https://github.com/coolguy860/Orislop-landing.git'
REPO_BRANCH = 'main'
DRIVE_BASE = '/content/drive/MyDrive/orislop-data'


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess
drive.mount('/content/drive')
token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add HF_TOKEN in Colab Secrets first')
os.environ['HF_TOKEN'] = token
repo = Path('/content/orislop')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(repo)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(repo / 'training/orislop_temporal_retrain/requirements-colab.txt')], check=True)


In [ ]:
drive_base = Path(DRIVE_BASE)
global_splits = drive_base / 'global-splits'
plan_root = drive_base / 'rolling-plan'
durable_root = drive_base / 'durable-caches'
base_config = repo / 'training/orislop_temporal_retrain/temporal_fusion_v5_config.example.json'
rolling = repo / 'training/orislop_dataset/rolling_pipeline.py'
if CREATE_PLAN:
    if plan_root.exists() and any(plan_root.iterdir()):
        raise RuntimeError(f'Plan directory is not empty: {plan_root}')
    subprocess.run(['python', str(rolling), 'plan', '--split-root', str(global_splits), '--output-root', str(plan_root), '--workers', '4', '--max-batch-gib', '50'], cwd=repo, check=True)
if not (plan_root / 'plan.json').is_file():
    raise FileNotFoundError('Create or upload rolling-plan/plan.json before starting workers')


In [ ]:
# Run this cell in all four runtimes with a different WORKER_ID.
subprocess.run([
    'python', str(rolling), 'run-worker',
    '--plan-root', str(plan_root),
    '--worker-id', str(WORKER_ID),
    '--stage-root', '/content/orislop-stage',
    '--durable-root', str(durable_root),
    '--base-config', str(base_config),
    '--execute', '--confirm-rights', '--cleanup',
], cwd=repo, check=True)


In [ ]:
# Safe to run at any time. Exit code 1 means some batches are unfinished.
subprocess.run(['python', str(rolling), 'status', '--plan-root', str(plan_root), '--durable-root', str(durable_root)], cwd=repo, check=False)


## Final synchronized retraining
Run the next cell only in worker 0 after status says `ready_to_merge: true`. It merges the four workers' caches, trains one fusion/router, calibrates on validation, evaluates on test, and packages the result.

In [ ]:
if WORKER_ID != 0:
    raise RuntimeError('Final merge/training runs only on worker 0')
merged_cache = drive_base / 'merged-expert-cache'
final_run = drive_base / 'final-temporal-run'
final_config = drive_base / 'final-temporal-config.json'
subprocess.run(['python', str(rolling), 'status', '--plan-root', str(plan_root), '--durable-root', str(durable_root)], cwd=repo, check=True)
subprocess.run(['python', str(rolling), 'merge-run', '--plan-root', str(plan_root), '--durable-root', str(durable_root), '--output-root', str(merged_cache)], cwd=repo, check=True)
subprocess.run(['python', str(rolling), 'render-final-config', '--base-config', str(base_config), '--merged-cache', str(merged_cache), '--run-root', str(final_run), '--output', str(final_config)], cwd=repo, check=True)
subprocess.run(['python', str(repo / 'training/orislop_temporal_retrain/colab_retrain.py'), 'run', '--config', str(final_config), '--start-at', 'train'], cwd=repo, check=True)
